# SmartClean Twin
## A Digital Twin of an Indoor Cleaning Robot

**Course:** RBB2013 Digital Twin
**Repository:** https://github.com/KAI-UTP/smartclean-twin
**Video:** https://youtu.be/zEq7L-ivMLA

| Name | Student ID |
|---|---|
| Chan Li Kai | 22010900 |
| Irvin Chang Hou Ceng | 22012342 |
| William Wong Xiao Kang | 22010943 |
| Nur Nemelin Melisa | 22011128 |
| Rene Rere | 22011952 |

---

### How to use this notebook

Every code cell is live. It queries the running system, not a saved file.
Run the setup cell first, then work down in order.

**Before starting:** run `START-SMARTCLEAN-TWIN.bat` so the nine containers are up.

---
## Setup

Run this once. It defines the connection settings and two helper functions used
throughout the notebook.

In [1]:
import subprocess, time, json, io, csv
import requests
import pandas as pd

INFLUX_URL    = "http://localhost:8086"
INFLUX_TOKEN  = "smartclean-super-secret-token"
INFLUX_ORG    = "smartclean"
INFLUX_BUCKET = "smartclean_twin"

SERVICES = {
    "Command API":      "http://localhost:8000/health",
    "State Engine":     "http://localhost:8002/health",
    "AI Service":       "http://localhost:8003/health",
    "Robot Simulator":  "http://localhost:8004/health",
    "Operator Console": "http://localhost:8005/health",
    "InfluxDB":         "http://localhost:8086/health",
    "Grafana":          "http://localhost:3001/api/health",
}

def flux(query):
    "Run a Flux query and return the result as a pandas DataFrame."
    r = requests.post(
        f"{INFLUX_URL}/api/v2/query?org={INFLUX_ORG}",
        data=query,
        headers={"Authorization": f"Token {INFLUX_TOKEN}",
                 "Content-Type": "application/vnd.flux",
                 "Accept": "application/csv"},
        timeout=30,
    )
    r.raise_for_status()
    rows = [x for x in csv.reader(io.StringIO(r.text)) if x and x[0] == ""]
    if not rows:
        return pd.DataFrame()
    header = rows[0][1:]
    data = [x[1:] for x in rows[1:] if x[1:] != header]
    return pd.DataFrame(data, columns=header)

def sh(cmd, timeout=60):
    "Run a shell command and print its output."
    p = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=timeout)
    print(p.stdout or p.stderr)

print("Ready.")

Ready.


---
# 1. The Project and the 5D Framework

### The problem

An indoor cleaning robot works alone in a building. The operator cannot see it,
cannot tell if it is stuck, and cannot tell if the motor is about to fail. A
camera would show what happened, not what is about to happen.

### What was built

A Digital Twin: a synchronised virtual copy that receives the robot's sensor
data, maintains its own understanding of the robot's condition, predicts what
happens next, and sends commands back.

### The distinction that matters

> A dashboard shows you data. A Digital Twin holds a synchronised model of the
> asset, reasons about it, predicts from it, answers hypothetical questions about
> it, and commands it back. This project does all five.

### The 5D framework

| Dimension | What it is | Where it lives here |
|---|---|---|
| **D1** Physical Entity | The asset | `services/robot-simulator/` |
| **D2** Virtual Model | Geometry, physics, behaviour, rules | `omniverse/create_scene.py`, `simulator.py`, `rules.py`, `predictor.py` |
| **D3** Services | What the twin does for you | `services/ai-service/`, `services/command-api/` |
| **D4** Data | What is stored, and how | InfluxDB, `shared/smartclean_common/models.py` |
| **D5** Connections | How the two halves talk | MQTT and REST, `shared/smartclean_common/topics.py` |

### The two engines

- **P2V** (physical to virtual): robot, MQTT, validation, database, views. Sections 3 to 6.
- **V2P** (virtual to physical): console, REST, MQTT, robot, acknowledgement. Section 9.

In [2]:
# The nine containers that make up the system
pd.DataFrame([
    ["robot-simulator",     8004, "Written",    "D1 Physical entity: physics, navigation, telemetry"],
    ["telemetry-ingestion", 8101, "Written",    "Validates every message, writes to the database"],
    ["state-engine",        8002, "Written",    "D2 Behaviour model: 11 twin state dimensions"],
    ["ai-service",          8003, "Written",    "D3 Service: 5 AI models and what-if simulation"],
    ["command-api",         8000, "Written",    "D3 Service: commands and acknowledgements"],
    ["web-control",         8005, "Written",    "Operator console, also works on a phone"],
    ["mosquitto",           1883, "Configured", "D5 Connection: the MQTT broker"],
    ["influxdb",            8086, "Configured", "D4 Data: the time series database"],
    ["grafana",             3001, "Configured", "Dashboard, 28 panels"],
], columns=["Container", "Port", "Origin", "Role"])

,Container,Port,Origin,Role
0,robot-simulator,8004,Written,"D1 Physical entity: physics, navigation, telem..."
1,telemetry-ingestion,8101,Written,"Validates every message, writes to the database"
2,state-engine,8002,Written,D2 Behaviour model: 11 twin state dimensions
3,ai-service,8003,Written,D3 Service: 5 AI models and what-if simulation
4,command-api,8000,Written,D3 Service: commands and acknowledgements
5,web-control,8005,Written,"Operator console, also works on a phone"
6,mosquitto,1883,Configured,D5 Connection: the MQTT broker
7,influxdb,8086,Configured,D4 Data: the time series database
8,grafana,3001,Configured,"Dashboard, 28 panels"


---
# 2. Docker: is the system running?

Nine containers, started by one command. Six are services written for this
project, three are infrastructure configured for it.

**File:** [`docker-compose.yml`](docker-compose.yml)

The startup ordering is not accidental. Mosquitto's healthcheck at line 28
actually subscribes to a topic rather than just checking the port is open, and
the other services declare `depends_on: condition: service_healthy`, so nothing
starts against a broker that is not ready.

In [3]:
sh('docker compose ps --format "table {{.Name}}\t{{.Status}}"')

NAME                                    STATUS
smartclean-ai-service                   Up 4 hours
smartclean-command-api                  Up 4 hours
smartclean-grafana                      Up 4 hours
smartclean-influxdb                     Up 4 hours (healthy)
smartclean-mosquitto                    Up 4 hours (healthy)
smartclean-simulator                    Up 4 hours
smartclean-state-engine                 Up 4 hours
smartclean-twin-telemetry-ingestion-1   Up 4 hours
smartclean-web-control                  Up 3 hours



In [4]:
# Health check on every service. All seven should say OK.
for name, url in SERVICES.items():
    try:
        r = requests.get(url, timeout=5)
        print(f"  [{'OK' if r.status_code == 200 else 'FAIL'}] {name}")
    except Exception as e:
        print(f"  [FAIL] {name}: {e}")

  [OK] Command API
  [OK] State Engine
  [OK] AI Service
  [OK] Robot Simulator
  [OK] Operator Console
  [OK] InfluxDB
  [OK] Grafana


---
# 3. MQTT: how the messages move

MQTT is a publish and subscribe protocol. Nobody sends a message *to a service*.
They publish to a **topic**, and every service subscribed to that topic gets a
copy. The publisher never learns who received it.

**File:** [`shared/smartclean_common/topics.py`](shared/smartclean_common/topics.py)

| Topic | Publisher | Subscribers |
|---|---|---|
| `telemetry/raw` | Simulator | Ingestion |
| `telemetry/validated` | Ingestion | State Engine, AI Service |
| `state` | State Engine | AI Service |
| `prediction` | AI Service | stored only |
| `command/motion`, `command/cleaning` | Command API | Simulator |
| `ack` | Simulator | Command API |

### Why MQTT and not REST for telemetry

The robot publishes once per second and does not know that three services are
listening. With REST, each of those services would have to poll the robot: three
times the load, and three slightly different answers to "where is the robot
now". MQTT sends one message and every subscriber sees the same instant.

The publish happens at **`simulator.py` line 513**, with QoS 1, meaning at least
once, so telemetry cannot silently vanish.

In [5]:
# Capture live MQTT traffic. The # is a wildcard for every topic below that point.
print("Listening for 8 messages on smartclean/SCR01/#\n")
sh('docker exec smartclean-mosquitto mosquitto_sub -t "smartclean/SCR01/#" -v -C 8 -W 15', timeout=40)

Listening for 8 messages on smartclean/SCR01/#



smartclean/SCR01/service/health {"service": "robot-simulator", "status": "healthy", "timestamp": "2026-08-12T22:36:13.158621+00:00", "uptime_s": 14432.5}
smartclean/SCR01/prediction {"robot_id": "SCR01", "timestamp": "2026-08-12T22:36:42.289231+00:00", "model_version": "1.0", "motor_health_prediction": "NORMAL", "motor_health_confidence": 0.95, "dirt_level_prediction": "CLEAN", "dirt_level_confidence": 1.0, "health_state_prediction": "NORMAL", "health_state_confidence": 0.9613, "predicted_rul_minutes": 111.5, "anomaly_score": 3.0442, "is_anomaly": false, "model_used": "random_forest", "minutes_to_empty": 981.0, "minutes_to_finish": 2.1, "recommendation": "Normal operation, no action needed"}
smartclean/SCR01/telemetry/raw {"schema_version": "1.0", "robot_id": "SCR01", "timestamp": "2026-08-12T22:36:43.161453+00:00", "sequence": 12068, "pose": {"x_m": 2.5, "y_m": 4.0, "heading_deg": 90.0, "speed_mps": 0.2}, "sensors": {"obstacle_cm": 200.0, "battery_v": 12.076, "battery_soc": 79.84, "ba

In [6]:
# One full telemetry message, formatted. This is the 16 field contract.
p = subprocess.run(
    'docker exec smartclean-mosquitto mosquitto_sub -t "smartclean/SCR01/telemetry/raw" -C 1 -W 10',
    shell=True, capture_output=True, text=True, timeout=30)
print(json.dumps(json.loads(p.stdout), indent=2))

{
  "schema_version": "1.0",
  "robot_id": "SCR01",
  "timestamp": "2026-08-12T22:36:45.161669+00:00",
  "sequence": 12070,
  "pose": {
    "x_m": 2.0,
    "y_m": 4.0,
    "heading_deg": 90.0,
    "speed_mps": 0.2
  },
  "sensors": {
    "obstacle_cm": 200.0,
    "battery_v": 12.076,
    "battery_soc": 79.84,
    "battery_a": 1.5,
    "motor_current_a": 0.9,
    "motor_temperature_c": 26.5,
    "dirt_score": 0.0227,
    "water_level_pct": 90.3,
    "bumper_active": false
  },
  "actuators": {
    "brush_on": true,
    "pump_on": false
  },
  "mission": {
    "mission_id": "MISSION-001",
    "mode": "CLEANING"
  },
  "_meta": {
    "cleaning_coverage_pct": 57.63,
    "cleaned_cells": 34,
    "total_accessible": 59
  }
}


---
# 4. Telemetry Ingestion: the validation gate

MQTT is the road. Ingestion is the checkpoint on that road. It is the only way
data enters the twin, and it has four jobs.

**File:** [`services/telemetry-ingestion/main.py`](services/telemetry-ingestion/main.py)

| Job | Line | What happens |
|---|---|---|
| 1. Parse and validate | 111, 119 | JSON parse, then check against the Pydantic schema |
| 2. Quarantine failures | 92 | Bad messages go to their own measurement, with the reason |
| 3. Store | 129 | Write 16 fields to InfluxDB |
| 4. Republish | 128 | Forward to `telemetry/validated` for downstream services |

### Two topics, not one

`raw` is what the robot said. `validated` is what the system accepted. Everything
downstream subscribes to `validated`, so validation happens exactly once, at the
boundary. Because they are separate topics, the rejection rate is measurable.

The schema is **`shared/smartclean_common/models.py` line 144**. Every field has
hard bounds: battery 0 to 100, heading 0 to under 360.

In [7]:
# Current ingestion statistics
port = subprocess.run("docker compose port telemetry-ingestion 8001",
                      shell=True, capture_output=True, text=True).stdout.strip()
ING = f"http://localhost:{port.rsplit(':', 1)[-1]}/health"
before = requests.get(ING, timeout=5).json()
print(json.dumps(before, indent=2))

{
  "status": "healthy",
  "uptime_s": 14294.7,
  "received": 14284,
  "valid": 14284,
  "invalid": 0,
  "valid_rate_pct": 100.0
}


### Live proof that the gate works

The next cell publishes a **deliberately invalid** message: a `battery_soc` of
999, which is outside the 0 to 100 bound in the schema.

Watch `invalid` increase by 1. The message is rejected and recorded, never stored
as if it were real.

In [8]:
bad = json.dumps({
    "schema_version": "1.0", "robot_id": "SCR01",
    "timestamp": "2026-08-13T10:00:00+00:00", "sequence": 1,
    "pose": {"x_m": 1.0, "y_m": 1.0, "heading_deg": 0.0, "speed_mps": 0.0},
    "sensors": {"obstacle_cm": 200.0, "battery_v": 12.0,
                "battery_soc": 999.0,          # invalid, schema allows 0 to 100
                "battery_a": 1.0, "motor_current_a": 0.7,
                "motor_temperature_c": 40.0, "dirt_score": 0.2,
                "water_level_pct": 80.0, "bumper_active": False},
    "actuators": {"brush_on": True, "pump_on": False},
    "mission": {"mission_id": "BAD-TEST", "mode": "CLEANING"},
})
subprocess.run(['docker', 'exec', 'smartclean-mosquitto', 'mosquitto_pub',
                '-t', 'smartclean/SCR01/telemetry/raw', '-m', bad],
               capture_output=True, text=True)
time.sleep(2)
after = requests.get(ING, timeout=5).json()

print(f"invalid before : {before['invalid']}")
print(f"invalid after  : {after['invalid']}   <-- rejected by the schema")
print(f"valid rate     : {after['valid_rate_pct']} %")

invalid before : 0
invalid after  : 1   <-- rejected by the schema
valid rate     : 99.99 %


In [9]:
# The rejected message is not lost. It is recorded with the reason why.
q = (f'from(bucket: "{INFLUX_BUCKET}") '
     f'|> range(start: -10m) '
     f'|> filter(fn: (r) => r._measurement == "robot_telemetry_invalid") '
     f'|> last()')
d = flux(q)
d[[c for c in ["_time", "reason", "_field", "_value"] if c in d.columns]]

,_time,reason,_field,_value
0,2026-08-12T22:36:45.97011Z,validation_error,count,1
1,2026-08-12T22:36:45.97011Z,validation_error,raw_preview,"{""schema_version"": ""1.0"", ""robot_id"": ""SCR01"",..."


---
# 5. InfluxDB: what is actually stored

InfluxDB is a time series database: every row has a timestamp, tags for
filtering, and fields for values.

**Connection:** built at **`telemetry-ingestion/main.py` line 54**. The
credentials come from environment variables injected by
**`docker-compose.yml` lines 7 to 13**. The URL is `http://influxdb:8086`, where
`influxdb` is the container name resolved by Docker's internal DNS. There is no
IP address anywhere in this codebase.

### Six measurements, kept deliberately separate

| Measurement | Written by | What it holds |
|---|---|---|
| `robot_telemetry` | Ingestion | What the sensors **measured** |
| `robot_state` | State Engine | What the twin **concluded** |
| `robot_prediction` | AI Service | What the AI **expects** |
| `robot_command` | Command API | Commands issued |
| `robot_acknowledgement` | Command API | What the robot confirmed |
| `robot_alarm` | State Engine | Threshold breaches |

Measurement, conclusion and prediction are three different kinds of truth. Mixing
them in one table means never being able to tell them apart again.

### Showing this in the InfluxDB web UI

Open **http://localhost:8086**, sign in as `admin` / `adminpassword`, then go to
**Data Explorer** and click **Script Editor**. The Data Explorer shows nothing
until a query is entered, which is why it can look empty.

Paste one of the queries from [`docs/influx-queries.md`](docs/influx-queries.md),
set the time range to **Past 15 minutes**, and press **Submit**.

The quickest one to show, robot position over time:

```
from(bucket: "smartclean_twin")
  |> range(start: -15m)
  |> filter(fn: (r) => r._measurement == "robot_telemetry")
  |> filter(fn: (r) => r._field == "x_m" or r._field == "y_m")
```

In [10]:
# Every measurement in the bucket
q = ('import "influxdata/influxdb/schema"\n'
     f'schema.measurements(bucket: "{INFLUX_BUCKET}")')
flux(q)[["_value"]].rename(columns={"_value": "measurement"})

,measurement
0,robot_acknowledgement
1,robot_alarm
2,robot_command
3,robot_prediction
4,robot_state
5,robot_telemetry
6,robot_telemetry_invalid


In [11]:
# Raw telemetry, last 2 minutes, one row per second with every field
q = (f'from(bucket: "{INFLUX_BUCKET}") '
     f'|> range(start: -2m) '
     f'|> filter(fn: (r) => r._measurement == "robot_telemetry") '
     f'|> pivot(rowKey: ["_time"], columnKey: ["_field"], valueColumn: "_value")')
df = flux(q)
cols = ["_time", "x_m", "y_m", "heading_deg", "battery_soc",
        "motor_temperature_c", "water_level_pct", "obstacle_cm"]
print(f"{len(df)} rows in the last 2 minutes, one per second\n")
df[[c for c in cols if c in df.columns]].tail(10)

120 rows in the last 2 minutes, one per second



,_time,x_m,y_m,heading_deg,battery_soc,motor_temperature_c,water_level_pct,obstacle_cm
110,2026-08-12T22:36:38.161575Z,3,4,90,79.85,26.5,90.3,200
111,2026-08-12T22:36:39.161688Z,3,4,90,79.85,26.5,90.3,200
112,2026-08-12T22:36:40.162Z,3,4,90,79.85,26.5,90.3,200
113,2026-08-12T22:36:41.161945Z,2.5,4,90,79.85,26.5,90.3,200
114,2026-08-12T22:36:42.16228Z,2.5,4,90,79.85,26.5,90.3,200
115,2026-08-12T22:36:43.16247Z,2.5,4,90,79.84,26.5,90.3,200
116,2026-08-12T22:36:44.162486Z,2.5,4,90,79.84,26.5,90.3,200
117,2026-08-12T22:36:45.162404Z,2,4,90,79.84,26.5,90.3,200
118,2026-08-12T22:36:46.162624Z,2,4,90,79.84,26.5,90.3,200
119,2026-08-12T22:36:47.162683Z,2,4,90,79.84,26.5,90.3,200


In [12]:
# How much data has accumulated, per measurement
for m in ["robot_telemetry", "robot_state", "robot_prediction",
          "robot_command", "robot_acknowledgement", "robot_alarm"]:
    q = (f'from(bucket: "{INFLUX_BUCKET}") '
         f'|> range(start: -1h) '
         f'|> filter(fn: (r) => r._measurement == "{m}") '
         f'|> count() |> group() |> sum()')
    d = flux(q)
    n = d["_value"].iloc[0] if len(d) else "0"
    print(f"  {m:<24} {n:>8} points in the last hour")

  robot_telemetry             57600 points in the last hour
  robot_state                 39600 points in the last hour
  robot_prediction            41213 points in the last hour
  robot_command                  28 points in the last hour


  robot_acknowledgement          28 points in the last hour
  robot_alarm                  1350 points in the last hour


---
# 6. Live Twin State: the robot's condition, as the twin understands it

Raw sensors are numbers. Twin state is meaning. The state engine converts one
into the other, across **11 dimensions**.

**File:** [`services/state-engine/rules.py`](services/state-engine/rules.py)

| Dimension | Line | Rule |
|---|---|---|
| Safety state | 59 | under 25 cm or bumper active is EMERGENCY, under 50 cm is WARNING |
| Battery state | 83 | under 10 percent CRITICAL, under 20 percent LOW |
| Motor health | 101 | over 70 C OVERHEATED, over 2.5 A HIGH_LOAD |
| Dirt level | 129 | 0.7 and above DIRTY, 0.3 and above MODERATE |
| Mission state | 161 | composite of coverage, battery and mode |
| Connection and twin quality | 179 | based on how old the last message is |

### The line to notice: 179

The twin knows how good it is. If telemetry is more than 2 seconds late it marks
itself DELAYED, and past 10 seconds INVALID. A twin that does not know it is
stale is worse than no twin, because the operator keeps trusting it.

In [13]:
# The twin's current state, all 11 dimensions
q = (f'from(bucket: "{INFLUX_BUCKET}") '
     f'|> range(start: -60s) '
     f'|> filter(fn: (r) => r._measurement == "robot_state") '
     f'|> last()')
s = flux(q)
s[["_field", "_value"]].rename(columns={"_field": "dimension", "_value": "value"})

,dimension,value
0,alarm_count,0
1,battery_state,NORMAL
2,cleaning_coverage_pct,59.32
3,connection_state,ONLINE
4,dirt_level,CLEAN
5,mission_state,RUNNING
6,motion_state,MOVING
7,motor_health,NORMAL
8,operation_mode,CLEANING
9,safety_state,SAFE


In [14]:
# Sensors measured, and the conclusion the twin drew from them, side by side
q = (f'from(bucket: "{INFLUX_BUCKET}") '
     f'|> range(start: -60s) '
     f'|> filter(fn: (r) => r._measurement == "robot_telemetry") '
     f'|> last()')
t = flux(q).set_index("_field")["_value"]
st = s.set_index("_field")["_value"]
g = lambda series, key: series.get(key, "--")

print("SENSOR MEASURED                          TWIN CONCLUDED")
print("-" * 64)
print(f"obstacle    {g(t,'obstacle_cm'):>12} cm          safety   : {g(st,'safety_state')}")
print(f"battery     {g(t,'battery_soc'):>12} %           battery  : {g(st,'battery_state')}")
print(f"motor temp  {g(t,'motor_temperature_c'):>12} C           motor    : {g(st,'motor_health')}")
print(f"dirt score  {g(t,'dirt_score'):>12}             dirt     : {g(st,'dirt_level')}")
print(f"{'':40}coverage : {g(st,'cleaning_coverage_pct')} %")
print(f"{'':40}quality  : {g(st,'twin_quality')}")

SENSOR MEASURED                          TWIN CONCLUDED
----------------------------------------------------------------
obstacle             200 cm          safety   : SAFE
battery            79.84 %           battery  : NORMAL
motor temp          26.5 C           motor    : NORMAL
dirt score             0             dirt     : CLEAN
                                        coverage : 59.32 %
                                        quality  : SYNCHRONIZED


---
# 7. Grafana and NVIDIA Omniverse: two views of one truth

Both read the **same InfluxDB**. That is exactly why they can never disagree.

### Grafana, http://localhost:3001/d/smartclean-main

28 panels in 7 sections. The datasource is
[`grafana/provisioning/datasources/influxdb.yaml`](grafana/provisioning/datasources/influxdb.yaml),
provisioned from a file in version control rather than clicked in the UI. If the
Grafana volume is deleted, `docker compose up` restores every panel exactly.

### NVIDIA Omniverse

- [`omniverse/create_scene.py`](omniverse/create_scene.py) builds the room, the
  100 coverage tiles and the robot, once.
- [`omniverse/live_update.py`](omniverse/live_update.py) streams the twin into
  the scene every second.

| Property | 3D prim | Source |
|---|---|---|
| Position and heading | `/World/CleaningRobot` | `x_m`, `y_m`, `heading_deg` |
| Body colour | `.../Body` | `safety_state` |
| Status light, flashing | `.../StatusLight` | `safety_state` |
| Battery bar | `.../BatteryBar` | `battery_soc` |
| Tile turns green | `/World/CoverageGrid/Tile_X_Y` | position |

### The question to expect: does the 3D scene use MQTT?

**No, and that is deliberate.** It reads InfluxDB, at `live_update.py` lines 87
and 102. The 3D view is a consumer, not a participant. It reads the same store
Grafana reads, which is why the two can never drift apart. Subscribing it to
MQTT directly would give it no history, and it would miss everything published
while Omniverse was closed.

The cost is latency: up to one second, because of the poll interval at line 64.
For a robot at 0.2 m/s that is 20 cm of error, which is acceptable here. For a
high speed asset I would subscribe to MQTT and keep the database for history.

In [15]:
# This is the exact query Omniverse runs every second (live_update.py line 102)
q = (f'from(bucket: "{INFLUX_BUCKET}") '
     f'|> range(start: -15s) '
     f'|> filter(fn: (r) => r._measurement == "robot_telemetry") '
     f'|> filter(fn: (r) => r._field == "x_m" or r._field == "y_m" or '
     f'r._field == "heading_deg" or r._field == "battery_soc") '
     f'|> last()')
flux(q)[["_field", "_value"]]

,_field,_value
0,battery_soc,79.84
1,heading_deg,90
2,x_m,2
3,y_m,4


In [16]:
# Grafana health, and the panel count in the dashboard file
print("Grafana:", requests.get("http://localhost:3001/api/health", timeout=5).json())
dash = json.load(open("grafana/dashboards/smartclean_twin.json", encoding="utf-8"))
print(f"\nDashboard '{dash.get('title')}': {len(dash.get('panels', []))} panels, provisioned from git")

Grafana: {'database': 'ok', 'version': '11.3.0', 'commit': 'd9455ff7db73b694db7d412e49a68bec767f2b5a'}

Dashboard 'SmartClean Twin - Robot Dashboard': 35 panels, provisioned from git


---
# 8. AI: five models, three learning paradigms

**Files:** [`services/ai-service/predictor.py`](services/ai-service/predictor.py)
at runtime, [`services/ai-service/train_model.py`](services/ai-service/train_model.py)
for training.

| Model | Type | Paradigm | Line in `predictor.py` |
|---|---|---|---|
| `motor_health_clf` | Random Forest | Supervised classification | 132 |
| `dirt_level_clf` | Random Forest | Supervised classification | 135 |
| `health_state_clf` | Random Forest | Supervised classification | 154 |
| `rul_regressor` | Random Forest | Supervised **regression** | 156 |
| `anomaly_detector` | mean and standard deviation per sensor | **Unsupervised** | 162 |

### Where the training data comes from

Generated in Python, in `train_model.py`, using the same sensor ranges as the
simulated robot. The physical robot does not exist, so there is no real history
to learn from. Generating it from the simulator's own ranges means the model
trains on the same distribution it sees in deployment. Everything is seeded at
42, so training is fully reproducible.

### The anomaly detector is the interesting one

It learned from **normal operation only** and never saw a fault during training.
It stores the mean and standard deviation of five sensors and flags anything more
than 4.5 standard deviations out. An injected 3.5 A motor current is 15 sigma
from the 0.75 A normal mean. It catches faults nobody labelled, which a
supervised classifier by definition cannot.

### Graceful degradation

If the model files are missing, `predict` falls back to rule based logic and
reports `model_used: rule_fallback`. The twin degrades, it does not crash, and it
says which path it took.

In [17]:
# Latest live prediction from the running AI service
q = (f'from(bucket: "{INFLUX_BUCKET}") '
     f'|> range(start: -60s) '
     f'|> filter(fn: (r) => r._measurement == "robot_prediction") '
     f'|> last()')
flux(q)[["_field", "_value"]]

,_field,_value
0,anomaly_score,3.0442
1,dirt_level,CLEAN
2,dirt_level_confidence,1
3,health_state,NORMAL
4,health_state_confidence,0.9613
5,is_anomaly,0
6,minutes_to_empty,980.9
7,minutes_to_finish,2
8,motor_health,NORMAL
9,motor_health_confidence,0.95


### What-if simulation: the defining capability of a Digital Twin

I can ask the virtual copy a question the real robot must never be asked. The
robot is not touched by either scenario below.

**Endpoint:** `ai-service/main.py` line 273.

In [18]:
def whatif(label, **kw):
    p = requests.post("http://localhost:8003/whatif", json=kw, timeout=15).json()["prediction"]
    print(f"\n{label}")
    print(f"  motor health   : {p['motor_health_prediction']}  (confidence {p['motor_health_confidence']})")
    print(f"  health state   : {p['health_state_prediction']}  (confidence {p['health_state_confidence']})")
    print(f"  remaining life : {p['predicted_rul_minutes']} minutes")
    print(f"  anomaly        : {p['is_anomaly']}  (score {p['anomaly_score']})")
    print(f"  recommendation : {p['recommendation']}")

whatif("SCENARIO A: healthy robot, 40 C at 0.8 A",
       motor_temperature_c=40.0, motor_current_a=0.8,
       battery_soc=90.0, water_level_pct=80.0)

whatif("SCENARIO B: what if the motor reached 95 C at 3.8 A?",
       motor_temperature_c=95.0, motor_current_a=3.8,
       battery_soc=40.0, water_level_pct=60.0)

print("\nThe real robot was never affected. A dashboard cannot do this.")


SCENARIO A: healthy robot, 40 C at 0.8 A


  motor health   : NORMAL  (confidence 0.99)
  health state   : NORMAL  (confidence 0.9046)
  remaining life : 108.8 minutes
  anomaly        : False  (score 3.8998)
  recommendation : Normal operation, no action needed

SCENARIO B: what if the motor reached 95 C at 3.8 A?
  motor health   : FAULT  (confidence 0.95)
  health state   : WARNING  (confidence 0.5404)
  remaining life : 24.6 minutes
  anomaly        : True  (score -12.5477)
  recommendation : Sensor anomaly detected: verify sensors and inspect robot

The real robot was never affected. A dashboard cannot do this.


---
# 9. User Control: the half that makes it a twin

A dashboard only displays. A twin also **commands, and confirms**.

**Console:** http://localhost:8005 (also works on a phone on the same WiFi)

### The full round trip, six files for one button press

```
app.js:184  ->  web-control/main.py:200  ->  command-api/main.py:215
                                                    |  MQTT
                                                    v
                                          simulator.py:84   receive
                                          simulator.py:97   apply
                                          simulator.py:227  publish ACK
                                                    |  MQTT
                                                    v
                                        command-api/main.py:104  receive
                                        command-api/main.py:231  unblocks
                                                    |  HTTP
                                                    v
                                                app.js:188
```

### The line that matters: `command-api/main.py:231`

```python
ack_received = ack_event.wait(timeout=ACK_TIMEOUT_S)
```

The HTTP response does not return until the robot has actually confirmed. The
operator is told what the robot **did**, not what the API hoped it would do. If
the robot is offline, the call honestly returns status timeout after 5 seconds.

### Manual driving

Eight directions, arrow keys, press and hold to repeat. A move into a wall is
**refused by the robot**, not by the web page, at `simulator.py` line 187.
Validation lives with the asset, which is where it belongs.

In [19]:
def command(cmd):
    r = requests.post("http://localhost:8000/api/v1/commands",
                      json={"robot_id": "SCR01", "command": cmd}, timeout=20).json()
    ok = r["ack_received"] and r.get("ack_accepted")
    print(f"  {cmd:<14} id={r['command_id']}  status={r['status']:<8} "
          f"{'ACCEPTED by the robot' if ok else 'NOT accepted'}")
    return r

print("Each response waits for the robot's real acknowledgement.\n")
command("PAUSE")
time.sleep(1)
command("RESUME")

Each response waits for the robot's real acknowledgement.

  PAUSE          id=CMD-66A1A439  status=acked    ACCEPTED by the robot


  RESUME         id=CMD-7B1636E4  status=acked    ACCEPTED by the robot


{'command_id': 'CMD-7B1636E4',
 'robot_id': 'SCR01',
 'command': 'RESUME',
 'status': 'acked',
 'request_timestamp': '2026-08-12T22:36:49.833702+00:00',
 'ack_received': True,
 'ack_timestamp': '2026-08-12T22:36:49.834553+00:00',
 'ack_accepted': True,
 'ack_reason': None}

In [20]:
# Manual control, including a move the robot will refuse
print("Taking manual control\n")
command("MANUAL_MODE")
time.sleep(1)

q = (f'from(bucket: "{INFLUX_BUCKET}") '
     f'|> range(start: -30s) '
     f'|> filter(fn: (r) => r._measurement == "robot_telemetry") '
     f'|> filter(fn: (r) => r._field == "x_m" or r._field == "y_m") '
     f'|> last()')
pos = flux(q).set_index("_field")["_value"]
print(f"position before: x={pos.get('x_m')} y={pos.get('y_m')}\n")

command("MOVE_RIGHT")
time.sleep(1)

print("\nNow driving north until the wall stops us:")
for _ in range(9):
    r = requests.post("http://localhost:8000/api/v1/commands",
                      json={"robot_id": "SCR01", "command": "MOVE_UP"}, timeout=20).json()
    if r.get("ack_accepted") is False:
        print("  MOVE_UP        REFUSED by the robot: a wall is in the way")
        print("  The robot turned to face it but did not move.")
        break
    time.sleep(0.4)

print("\nReturning to autonomous")
command("AUTO_MODE")
print("\nIt resumes from where it is, it does not teleport back. (simulator.py:161)")

Taking manual control

  MANUAL_MODE    id=CMD-4198CDC8  status=acked    ACCEPTED by the robot


position before: x=2 y=4

  MOVE_RIGHT     id=CMD-F8472B3B  status=acked    ACCEPTED by the robot



Now driving north until the wall stops us:
  MOVE_UP        REFUSED by the robot: a wall is in the way
  The robot turned to face it but did not move.

Returning to autonomous
  AUTO_MODE      id=CMD-8EF29B9F  status=acked    ACCEPTED by the robot

It resumes from where it is, it does not teleport back. (simulator.py:161)


---
# 10. GitHub and CI: how the code is managed

**Repository:** https://github.com/KAI-UTP/smartclean-twin

### Branching

Git Flow. `main` is always deployable, `develop` integrates, and there is one
feature branch per service, matching the microservice boundaries.

### Continuous Integration

**File:** [`.github/workflows/ci.yml`](.github/workflows/ci.yml)

Every push to GitHub automatically runs five checks. A green tick means all
passed. Nothing is run by hand.

```
lint-and-format
     |--> unit-tests --> build-docker-images --> validate-compose
     `--> train-ai-model
```

| Check | What it verifies |
|---|---|
| 1. Lint and format | ruff and black, versions pinned |
| 2. Tests | every unit, integration and regression test, with coverage |
| 3. **Train AI model** | retrains all 5 models, **fails if accuracy drops below 80 percent** |
| 4. Build images | all 5 Docker images build on a clean machine |
| 5. Validate compose | `docker compose config` parses and resolves |

### Check 3 is the unusual one

Most projects test their code and trust their model. If someone changed a feature
mapping and accuracy quietly fell from 90 percent to 70, ordinary CI would still
go green, because the code still runs. This job catches it. The gates are at
`train_model.py` lines 398 to 406.

### CI has actually caught things

- An unpinned linter released a stricter version and failed a **documentation
  only** commit. Fixed by pinning versions.
- A change dropped coverage to 64 percent against a 65 percent gate. Rather than
  lower the gate, 44 new tests were written and coverage reached 82 percent.

> A quality gate you lower when it becomes inconvenient is not a quality gate.

In [21]:
print("Commits:", subprocess.run("git rev-list --count HEAD", shell=True,
      capture_output=True, text=True).stdout.strip())
print("\nBranches:")
sh("git branch")
print("Recent history, note the conventional commit prefixes:")
sh("git log --oneline -10")

Commits: 69

Branches:
  develop
  feature/ai-service
  feature/command-api
  feature/grafana
  feature/ingestion
  feature/simulator
  feature/state-engine
  feature/testing-ci
* main

Recent history, note the conventional commit prefixes:


ba6b2c8 Add complete 0-100 walkthrough and fix smoke test Grafana port
12c32dd Add presentation deep dive with file and line references for every claim
5c05d25 Fix launcher: open InfluxDB, swap PDF for the walkthrough notebook, repair Omniverse launch
42eb121 fix: launcher would not run, and the ingestion port range stole the AI service port
752a683 docs: rehearsed demonstration runbook, and refuse to tunnel without a password
8a81ed9 fix: issues found in a full self audit of the codebase
c2c3ee6 fix: robot no longer jumps across the room when leaving manual mode
71e8dbc feat: phone access through a tunnel, with an optional password gate
0eac2f3 feat: eight-direction driving, quick action macros and phone access
875539d feat: robot returns to the dock for water as well as for battery



In [22]:
# The test suite, run live
sh("py -m pytest tests/unit -q --no-header", timeout=900)

============================= test session starts =============================
collected 169 items

tests\unit\test_ai_predictor.py .........                                [  5%]
tests\unit\test_command_validation.py .............                      [ 13%]
tests\unit\test_grid_map.py ..........                                   [ 18%]
tests\unit\test_manual_control.py .............................          [ 36%]
tests\unit\test_return_to_dock.py ...............                        [ 44%]
tests\unit\test_simulator_commands.py ............                       [ 52%]
tests\unit\test_state_rules.py ....................                      [ 63%]
tests\unit\test_telemetry_schema.py .................                    [ 73%]
tests\unit\test_web_control.py ......................................... [ 98%]
...                                                                      [100%]

============================== warnings summary ===============================
tests/unit/test_ma

---
# 11. Limitations and Further Improvements

Three real limitations, each located in the code. Stating them is better than
having them found.

### 1. Single robot assumption

`state-engine/main.py` lines 44 to 46 keep coverage and sequence in module level
globals, so a second robot would overwrite the first. The topic hierarchy at
`topics.py` line 7 already anticipates a fleet (`smartclean/SCR02/...`), but the
state engine does not yet.

**Fix:** key the state by `robot_id` in a dictionary. The messaging layer needs
no change at all.

### 2. The 3D view polls instead of subscribing

Up to one second behind, because of the poll interval at `live_update.py` line 64.

**Fix:** subscribe to MQTT for live position and keep InfluxDB for history. The
trade off was made deliberately, not by accident.

### 3. Thresholds are fixed constants

Every threshold in `rules.py` is hard coded: 70 C, 2.5 A, 20 percent. Real assets
differ from each other and drift as they age.

**Fix:** learn per asset baselines, the same way the anomaly detector already
learns its own.

### Beyond that

| Improvement | Why |
|---|---|
| Fleet coordination | Several robots sharing one floor plan, dividing the work |
| Real hardware | Replace the simulator with an actual robot |
| Longer horizon prediction | Current RUL is minutes. Days would need months of history |
| Alert delivery | Alarms are stored and displayed, but not pushed to anyone |

### Why the architecture is already ready for real hardware

No service imports the simulator. They subscribe to a topic. Replacing the
simulated robot with a physical one that publishes the same schema requires
changes to **zero** other services. That is the payoff from validating at the
boundary and defining the contract in one file.

---

## Summary

| | |
|---|---|
| Containers | 9, six of them services written for this project |
| Telemetry | 16 fields, once per second |
| Twin state | 11 dimensions |
| AI models | 5, across 3 learning paradigms |
| Health classifier | 90.8 percent accuracy |
| RUL regressor | R squared 0.91, mean error 6.6 minutes |
| Anomaly detector | 100 percent of known faults, no false alarms |
| Grafana | 28 panels in 7 sections |
| Tests | 204, coverage 82 percent |
| CI | 5 automated checks on every push |

**Repository:** https://github.com/KAI-UTP/smartclean-twin
**Video:** https://youtu.be/zEq7L-ivMLA

### Thank you. Questions?